# Evaluation — Hybrid Log Analyzer

Loads the trained GAT model, runs inference on the test set, computes metrics
(AUROC, AUPRC, F1), generates plots (ROC, loss curves), and writes
`metrics.json` for DVC and CML consumption.

In [ ]:
# ── Parameters (injected by Papermill) ────────────────────
dataset_path = "data/processed"
checkpoint_path = "models/gat_model.pt"
metrics_output = "metrics.json"
plots_dir = "plots"
reconstruction_threshold = 0.5
random_seed = 42

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath("."))

from src.utils import seed_everything, get_device, is_ci
seed_everything(random_seed)
device = get_device()
print(f"Evaluating on: {device}  |  CI={is_ci()}")

In [ ]:
from pathlib import Path
import json
import numpy as np
from src.model import load_checkpoint
from src.evaluate import compute_metrics, plot_roc_curve, plot_loss_curve, save_metrics

# Load checkpoint
print(f"Loading checkpoint from {checkpoint_path}")
ckpt = load_checkpoint(checkpoint_path)
print(f"Checkpoint at epoch {ckpt['epoch']} — loss {ckpt['loss']:.4f}")

In [ ]:
# ── Generate evaluation metrics ─────────────────────────
# TODO: Replace with real inference once data pipeline is complete
#       Load test data, run model.encode(), compute reconstruction error,
#       threshold to get predictions.

# Placeholder: synthetic predictions for pipeline validation
np.random.seed(random_seed)
n_samples = 200
y_true = np.concatenate([np.zeros(n_samples // 2), np.ones(n_samples // 2)])
y_scores = np.concatenate([
    np.random.beta(2, 5, n_samples // 2),   # normal: low scores
    np.random.beta(5, 2, n_samples // 2),    # anomaly: high scores
])

metrics = compute_metrics(y_true, y_scores)
metrics["reconstruction_threshold"] = reconstruction_threshold
metrics["checkpoint_epoch"] = ckpt["epoch"]
metrics["train_loss"] = round(float(ckpt["loss"]), 6)
print(f"Metrics: {json.dumps(metrics, indent=2)}")

In [ ]:
# ── Generate plots ──────────────────────────────────────
Path(plots_dir).mkdir(parents=True, exist_ok=True)

plot_roc_curve(y_true, y_scores, out_path=f"{plots_dir}/roc.png")
print(f"ROC curve saved to {plots_dir}/roc.png")

# Load training history if available
history_path = Path("outputs/train_history.json")
if history_path.exists():
    with open(history_path) as f:
        history = json.load(f)
    plot_loss_curve(
        history["train_losses"],
        history.get("val_losses"),
        out_path=f"{plots_dir}/loss.png",
    )
    print(f"Loss curve saved to {plots_dir}/loss.png")
else:
    print("No training history found — skipping loss curve")

In [ ]:
# ── Save metrics for DVC & CML ──────────────────────────
save_metrics(metrics, metrics_output)
print("Evaluation complete.")